In [3]:
import os
from dotenv import load_dotenv
from unstructured.partition.pdf import partition_pdf

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGCHAIN_TRACING_V2 = "true"


output_path = "../output/"
file_path = "../content/LGUC.pdf"

In [4]:
from unstructured.partition.pdf import partition_pdf

# Reference: https://docs.unstructured.io/open-source/core-functionality/chunking
chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,            # extract tables
    strategy="hi_res",                     # mandatory to infer tables

    extract_image_block_types=["Image"],   # Add 'Table' to list to extract image of tables
    # image_output_dir_path=output_path,   # if None, images and tables will saved in base64

    extract_image_block_to_payload=True,   # if true, will extract base64 for API usage

    chunking_strategy="by_title",          # or 'basic'
    max_characters=10000,                  # defaults to 500
    combine_text_under_n_chars=2000,       # defaults to 0
    new_after_n_chars=6000,

    # extract_images_in_pdf=True,          # deprecated
)

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

In [5]:
len(chunks)

110

In [6]:
elements = chunks[20].metadata.orig_elements

print("\n\nChunk")
for e in elements:
    print(e.to_dict()["type"], e.metadata.page_number)




Chunk
Title 19
NarrativeText 19
Title 19
Title 19
NarrativeText 19
Title 19
NarrativeText 19
Title 20
Image 20
Header 20
NarrativeText 20
NarrativeText 20
NarrativeText 20
NarrativeText 20
NarrativeText 20
NarrativeText 20
NarrativeText 20
NarrativeText 20


In [7]:
texts = []
for chunk in chunks:
    texts.append(chunk)

In [11]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")

def count_tokens(text: str) -> int:
    """Cuenta los tokens en un texto usando el tokenizer de Llama"""
    return len(tokenizer.encode(text))

prompt_text = """
Eres un asistente que resume tablas y texto.
Entrega un resumen conciso de la tabla o texto.

No comiences tu mensaje diciendo "Aqui hay un resumen" o algo similar.
Simplemente entrega el resumen como tal.

Tabla o texto: {element}
"""
base_token_count = count_tokens(prompt_text.replace("{element}", ""))
print(f"Tokens en el template base: {base_token_count}")

prompt = ChatPromptTemplate.from_template(prompt_text)

def process_with_token_count(x):
    tokens = count_tokens(x)
    total_tokens = base_token_count + tokens
    print(f"Tokens en el input: {tokens}")
    print(f"Tokens totales en el request: {total_tokens}")
    return x

# Summary chain con token count
model = ChatGroq(temperature=0.5, model="llama-3.1-8b-instant")
summarize_chain = {"element": process_with_token_count} | prompt | model | StrOutputParser()

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Tokens en el template base: 81


In [13]:
# Extraer el texto de los elementos antes de procesarlos
text_contents = [text.text for text in texts[:2]]  # Accedemos a la propiedad .text de cada elemento
text_summaries = summarize_chain.batch(text_contents, {"max_concurrency": 3})

Tokens en el input: 892
Tokens totales en el request: 973
Tokens en el input: 728
Tokens totales en el request: 809


In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_text = """
Eres un asistente que resume tablas y texto.
Entrega un resumen conciso de la tabla o texto.

No comiences tu mensaje diciendo "Aqui hay un resumen" o algo similar.
Simplemente entrega el resumen como tal.

Tabla o texto: {element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)

# Summary chain
model = ChatGroq(temperature=0.5, model="llama-3.1-8b-instant")
summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()

In [22]:
import asyncio
from datetime import datetime, timedelta
from collections import deque
import time
from typing import List, Any
import nest_asyncio

class RateLimiter:
    def __init__(self):
        self.rpm_limit = 1000  # Requests per minute
        self.tpm_limit = 250000  # Tokens per minute
        self.rpd_limit = 500000  # Requests per day
        
        # Ventanas deslizantes para tracking
        self.minute_requests = deque(maxlen=1000)
        self.minute_tokens = deque(maxlen=250000)
        self.day_requests = deque(maxlen=500000)
        
        # Tiempo mínimo entre requests (60 seg / 1000 rpm = 0.06 seg)
        self.min_interval = 60 / self.rpm_limit

    async def wait_if_needed(self, tokens: int = 1):
        current_time = datetime.now()
        
        # Limpiar registros antiguos
        self._clean_old_records(current_time)
        
        # Verificar límites
        while (
            len(self.minute_requests) >= self.rpm_limit or
            len(self.minute_tokens) + tokens >= self.tpm_limit or
            len(self.day_requests) >= self.rpd_limit
        ):
            await asyncio.sleep(0.1)
            current_time = datetime.now()
            self._clean_old_records(current_time)
        
        # Registrar nueva solicitud
        self.minute_requests.append(current_time)
        self.day_requests.append(current_time)
        for _ in range(tokens):
            self.minute_tokens.append(current_time)

    def _clean_old_records(self, current_time: datetime):
        minute_ago = current_time - timedelta(minutes=1)
        day_ago = current_time - timedelta(days=1)
        
        while (self.minute_requests and 
               self.minute_requests[0] < minute_ago):
            self.minute_requests.popleft()
            
        while (self.minute_tokens and 
               self.minute_tokens[0] < minute_ago):
            self.minute_tokens.popleft()
            
        while (self.day_requests and 
               self.day_requests[0] < day_ago):
            self.day_requests.popleft()

async def process_batch_with_rate_limit(items: List[Any], 
                                      process_func, 
                                      batch_size: int = 3,
                                      tokens_per_request: int = 1000):
    """
    Procesa items en batches respetando rate limits
    """
    rate_limiter = RateLimiter()
    results = []
    
    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]
        batch_tasks = []
        
        for item in batch:
            # Esperar si es necesario por rate limits
            await rate_limiter.wait_if_needed(tokens_per_request)
            # Crear y agregar la tarea
            task = asyncio.create_task(process_func(item))
            batch_tasks.append(task)
        
        # Esperar que se complete el batch actual
        batch_results = await asyncio.gather(*batch_tasks)
        results.extend(batch_results)
        
    return results

# Ejemplo de uso con tu código existente
async def main():
    # Para textos
    text_summaries = await process_batch_with_rate_limit(
        items=texts,
        process_func=summarize_chain.ainvoke,
        batch_size=3,
        tokens_per_request=1000  # Ajusta según el promedio de tokens por request
    )
    
#     # Para tablas
#     tables_content = []
#     for table in tables:
#         table_html = table.metadata.text_as_html 
#         table_text = table.text
#         element = table_html + "\n\n" + table_text
#         tables_content.append(element)
    
#     tables_summaries = await process_batch_with_rate_limit(
#         items=tables_content,
#         process_func=summarize_chain.ainvoke,
#         batch_size=3,
#         tokens_per_request=1000
#     )

text_contents = [text.text for text in texts]

# # Ejecutar
async def process_summaries():
    text_summaries = await process_batch_with_rate_limit(
    items=text_contents,  # Usamos text_contents en lugar de texts
    process_func=lambda x: summarize_chain.ainvoke({"element": x}),  # Corregimos el formato
    batch_size=3,
    tokens_per_request=1000
    )
    return text_summaries

nest_asyncio.apply()

text_summaries = await process_summaries()




In [23]:
text_summaries

['Se aprueba la nueva Ley General de Urbanismo y Construcciones a través del Decreto 458, emitido por el Ministerio de Vivienda y Urbanismo en 1975. La ley establece principios, atribuciones y responsabilidades para organismos, funcionarios y particulares en materia de planificación urbana, urbanización y construcción. También regula el procedimiento administrativo y los estándares técnicos de diseño y construcción. La ley tiene un nivel de acción general y se aplica en todo el territorio nacional.',
 'El Ministerio de Vivienda y Urbanismo tiene las siguientes responsabilidades:\n\n- Proponer modificaciones a la Ley para adecuarla al desarrollo nacional.\n- Estudiar y aprobar modificaciones a la Ordenanza General de la Ley para mantenerla al día con el avance tecnológico y desarrollo socio-económico.\n- Aprobar por decreto supremo las Normas Técnicas del Instituto Nacional de Normalización y las normas sobre pavimentación.\n- Supervigilar las disposiciones legales, reglamentarias, admi

In [24]:
len(text_summaries)

110

In [25]:
import uuid
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain.retrievers.multi_vector import MultiVectorRetriever

vectorstore = Chroma(collection_name="multi_modal_rag", embedding_function=OpenAIEmbeddings())

store = InMemoryStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

/tmp/ipykernel_107072/3994283437.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(collection_name="multi_modal_rag", embedding_function=OpenAIEmbeddings())


In [26]:
# Add texts
doc_ids = [str(uuid.uuid4()) for _ in texts]
summary_texts = [
    Document(page_content=summary, metadata={id_key: doc_ids[i]}) for i, summary in enumerate(text_summaries)
]
retriever.vectorstore.add_documents(summary_texts)
retriever.docstore.mset(list(zip(doc_ids, texts)))

In [27]:
chunks = retriever.invoke(
    "que dice la ley con respecto a los estacionamientos?"
)

print(type(chunks))

<class 'list'>


In [28]:
chunks

In [29]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from base64 import b64decode


def parse_docs(docs):
    """Split base64-encoded images and texts"""
    b64 = []
    text = []
    for doc in docs:
        try:
            b64decode(doc)
            b64.append(doc)
        except Exception as e:
            text.append(doc)
    return {"images": b64, "texts": text}


def build_prompt(kwargs):

    docs_by_type = kwargs["context"]
    user_question = kwargs["question"]

    context_text = ""
    if len(docs_by_type["texts"]) > 0:
        for text_element in docs_by_type["texts"]:
            context_text += text_element.text

    # construct prompt with context (including images)
    prompt_template = f"""
    Answer the question based only on the following context, which can include text, tables, and the below image.
    Context: {context_text}
    Question: {user_question}
    """

    prompt_content = [{"type": "text", "text": prompt_template}]

    if len(docs_by_type["images"]) > 0:
        for image in docs_by_type["images"]:
            prompt_content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{image}"},
                }
            )

    return ChatPromptTemplate.from_messages(
        [
            HumanMessage(content=prompt_content),
        ]
    )


chain = (
    {
        "context": retriever | RunnableLambda(parse_docs),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(build_prompt)
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

chain_with_sources = {
    "context": retriever | RunnableLambda(parse_docs),
    "question": RunnablePassthrough(),
} | RunnablePassthrough().assign(
    response=(
        RunnableLambda(build_prompt)
        | ChatOpenAI(model="gpt-4o-mini")
        | StrOutputParser()
    )
)

In [32]:
response = chain_with_sources.invoke(
    "que dice la ley con respecto al uso de suelo?"
)

print("Response:", response['response'])

print("\n\nContext:")
for text in response['context']['texts']:
    print(text.text)
    print("Page number: ", text.metadata.page_number)
    print("\n" + "-"*50 + "\n")

Response: La ley establece que el uso del suelo urbano en las áreas urbanas se regirá por las disposiciones de los Planes Reguladores. Las construcciones que se realicen en estos terrenos deben ser concordantes con el propósito de dichos planes. Además, el otorgamiento de patentes municipales también debe ser consistente con el uso del suelo establecido en la planificación urbana. Cualquier patente que vulnere este uso del suelo causará la caducidad automática de la misma y puede resultar en la destitución del funcionario que la otorgó. También se declara de utilidad pública todos los terrenos mencionados en los planes reguladores destinándolos para circulaciones, plazas, parques y vialidades en áreas urbanas y rurales.


Context:
Art. ÚNICO N° 1 a)

D.O. 19.08.2016

Ley 20943

Art. ÚNICO N° 1 a)

D.O. 19.08.2016

Artículo 56°.- En las áreas rurales, se prohíbe a los dueños de predios colindantes con los caminos públicos nacionales, definidos por la Ley de Caminos, ocupar las franjas d